# ROCIO-IBEB - series areales
***

***Author:** Javier Díez Sierra*<br>
***Date:** 19-05-2026*<br>

**Introducción:**<br>
En este _notebook_ se calculan las series meteorológicas areales de las cuencas de CAMELS-ES a partir de los datos meteorológicos de [ROCIO-IBEB](https://www.aemet.es/es/serviciosclimaticos/cambio_climat/datos_diarios/ayuda/rejilla_5km). Los datos han sido descargados de la página oficial de AEMET: https://www.aemet.es/es/serviciosclimaticos/cambio_climat/datos_diarios?w=1 

Las series de ROCIO-IBEB empiezan el 1 de enero de 1950 y terminan el 31 de diciembre de 2022. Ya que las series de aforos comienzan el 1 de enero de 1979, usaremos dicho periodo como fecha inicial.

En este notebook estamos usando datos de ROCIO-IBEB interpolados a una malla regular de 5x5km, pero se podría trabajar con la proyección original en la malla rotada usando el paquete rio, aunque no deberían existir muchas diferencias en los resultados al tratarse de datos areales medios en sub-cuencas.

In [1]:
import geopandas as gpd
import xarray as xr
import xclim
import logging
logger = logging.getLogger(__name__)

from ocab.config import Config
from ocab.basins.stats import read_data, read_pixarea, basin_statistics

import warnings
warnings.filterwarnings('ignore', message='invalid value encountered in sqrt', category=RuntimeWarning)

In [2]:
import numpy as np
from typing import Optional

def compute_pixel_area(ds: xr.Dataset, proj: Optional[str] = None) -> xr.DataArray:
    """Computes the physical area (in square meters) of each pixel in a 
    rotated-pole grid using its 2D geographical lat/lon coordinates.
    """
    # 1. Define Earth's radius in meters
    if proj:
        # calculate the Authalic Radius (Equal-Area Sphere Radius)
        attrs = ds[proj].attrs
        a = attrs['semi_major_axis']
        f = 1 / attrs['inverse_flattening']
        e = np.sqrt(2 * f - f**2) # eccentricity
        R = a * np.sqrt(0.5 + (1 - e**2) / (4 * e) * np.log((1 + e) / (1 - e)))
    else:
        R = 6371000.0 

    # 2. Get resolution in radians
    d_rlat = np.abs(np.gradient(ds['rlat']))[0]
    d_rlon = np.abs(np.gradient(ds['rlon']))[0]
    
    d_lat_rad = np.radians(d_rlat)
    d_lon_rad = np.radians(d_rlon)

    # 3. Convert 2D geographic latitude array to radians
    lat_rad = np.radians(ds['lat'])

    # 4. Spherical grid cell area formula: Area = R² * cos(lat) * d_lat * d_lon
    area = (R**2) * np.cos(lat_rad) * d_lat_rad * d_lon_rad

    # 5. Clean up attributes
    area.name = 'area'
    area.attrs['units'] = 'm2'
    area.attrs['long_name'] = 'Pixel Area'

    area = area.rio.write_crs(ds.rio.crs)
    
    return area

## Configuration

In [3]:
# dataset configuration
cfg = Config('./config_CAMELS_v200.yml')

# basins shapefile
basins_file = cfg.path_dataset / 'preprocessing' / 'basins' / 'output' / 'stations_basins_3sec.geojson'

# meteorology
meteo = 'ROCIO-IBEB'
zarr_store = f'{meteo}_1979-2022.zarr'
pet_method = 'HG85' # 'HG85': Hargreaves, 'DA02': Droogers-Allen

## Data

In [4]:
# load basins shapefile
if basins_file.is_file():
    basins = gpd.read_file(basins_file).set_index('ID')
else:
    logger.error(f"Basins file doesn't exist: {basins_file}")

In [5]:
# load meteorological data
cfg.path_meteo = cfg.path_meteo.parent.parent / meteo
zarr_store = cfg.path_meteo / zarr_store
if zarr_store.is_dir():
    data = read_data(zarr_store)
    print(f"{data.nbytes / 1e9:.2f} GB")
    # rewrite temperature units
    for var in ['mintemp', 'maxtemp']:
        data[var].attrs['units'] = 'degC'
    # rewrite precipitation units
    data['precipitation'].attrs['units'] = 'mm/d'
else:
    logger.error(f"Zarr store doesn't exist: {zarr_store}")

12.96 GB


## Compute

### Potential Evapotranspiration

In [6]:
if pet_method == 'HG85':
    pet = xclim.indices.potential_evapotranspiration(
        tasmin=data['mintemp'], 
        tasmax=data['maxtemp'], 
        lat=data['lat'],
        method=pet_method
    )
elif pet_method == 'DA02':
    pet = xclim.indices.potential_evapotranspiration(
        tasmin=data['mintemp'], 
        tasmax=data['maxtemp'], 
        pr=data['precipitation'],
        lat=data['lat'],
        method=pet_method
        )
else:
    logger.error(f"`pet_method` must be either 'HG85' or 'DA02'; {pet_method} was provided.")

# convert from kg m-2 s-1 to mm/d
pet *= 86400
pet.attrs['units'] = 'mm/d'
pet.attrs['long_name'] = f'Potential Evapotranspiration ({pet_method})'

# add to dataset
data['e0'] = pet

/home/casadoj/Git/of_camels_and_beavers/.venv/lib/python3.12/site-packages/dask/array/core.py:5198: PerformanceWarning: Increasing number of chunks by factor of 42
  result = blockwise(


### Pixel Area

In [7]:
pixarea = compute_pixel_area(data, proj='rotated_pole').compute()

### Basin Statistics

In [8]:
# compute statistics
vars = ['precipitation', 'mintemp', 'maxtemp', 'e0']
results = basin_statistics(
    data=data[vars],
    basins=basins,
    statistic='mean',
    weight=pixarea,
    decimals=1,
    output=cfg.path_dataset / 'preprocessing' / 'timeseries' / 'meteo' / meteo
)

basins:   0%|          | 0/1117 [00:00<?, ?it/s]